# Stream to Shorts — Google Colab

Run Cell 1 once, restart the runtime, then run Cell 2. Your YouTube URL and clip settings are preserved.

In [ ]:
# CELL 1 — RUN THIS ONCE, THEN RESTART THE COLAB RUNTIME
# Installs Faster-Whisper with its required dependencies and compatible PyAV.
import sys, subprocess

packages = [
    "av==18.1.0",
    "faster-whisper==1.2.1",
    "ctranslate2==4.9.0",
    "huggingface-hub<2",
    "numpy<2.5",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir", *packages
])

print("Dependencies installed successfully.")
print("NOW: Runtime > Restart session. Then run CELL 2.")


After Cell 1 finishes, choose **Runtime → Restart session**. Then run Cell 2.

In [ ]:
# CELL 2 — AFTER RESTART, RUN THIS ENTIRE CELL
# Do not reinstall PyAV or Faster-Whisper in this cell.


# ============================================================
# AI STREAM TO SHORTS GENERATOR
# Google Colab | Long YouTube videos | Maximum 10 clips
# ============================================================

# 1. SETTINGS
YOUTUBE_URL = "https://www.youtube.com/live/hwSg39yIBGU?si=H_RkA6PX5w1lcaqb"

TOTAL_CLIPS = 10
CLIP_DURATION = 45
CHUNK_MINUTES = 15
VIDEO_HEIGHT = 720
WHISPER_MODEL = "small"  # tiny, base, small, medium

# ============================================================
# 2. INSTALL ONLY YT-DLP AND FFMPEG
# Do NOT reinstall PyAV or Faster-Whisper here.
# ============================================================

import os
import sys
import glob
import json
import time
import zipfile
import subprocess
from pathlib import Path
from bisect import bisect_left

print("Installing required video tools...")

subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg",
    shell=True,
    check=True
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp"],
    check=True
)

# ============================================================
# 3. VERIFY COMPATIBLE AI LIBRARIES
# ============================================================
import av
import torch
from faster_whisper import WhisperModel

assert av.__version__ == "18.1.0", (
    f"Incorrect PyAV version: {av.__version__}"
)

print("Compatible AI libraries loaded!")
print("PyAV:", av.__version__)
print("Faster-Whisper is ready.")

# ============================================================
# 4. GOOGLE DRIVE
# ============================================================

from google.colab import drive, files

drive.mount("/content/drive")

WORK = Path("/content/stream_clipper")
SAVE = Path("/content/drive/MyDrive/Stream_Clips")

WORK.mkdir(parents=True, exist_ok=True)
SAVE.mkdir(parents=True, exist_ok=True)

CHECKPOINT = SAVE / "transcription_checkpoint.json"
ZIP_FILE = SAVE / "Stream_Viral_Clips.zip"

AUDIO_TEMPLATE = SAVE / "stream_audio.%(ext)s"

# Find an existing completed audio download.
def find_audio():
    matches = glob.glob(str(SAVE / "stream_audio.*"))
    matches = [
        p for p in matches
        if not p.endswith(".part")
        and not p.endswith(".ytdl")
    ]
    return Path(matches[0]) if matches else None

AUDIO = find_audio()

# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def run_command(command):
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr[-4000:])

    return result


def get_duration(filename):
    result = run_command([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(filename)
    ])

    return float(result.stdout.strip())


def srt_time(seconds):
    milliseconds = int(max(0, seconds) * 1000)

    hours = milliseconds // 3600000
    minutes = (milliseconds % 3600000) // 60000
    seconds = (milliseconds % 60000) // 1000
    ms = milliseconds % 1000

    return (
        f"{hours:02}:{minutes:02}:"
        f"{seconds:02},{ms:03}"
    )


def save_checkpoint(data):
    temp = CHECKPOINT.with_suffix(".tmp")

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(data, f)

    os.replace(temp, CHECKPOINT)


# ============================================================
# 6. DOWNLOAD AUDIO
# Audio is saved permanently in Google Drive.
# ============================================================

if AUDIO is None:
    print("\nDownloading YouTube audio...")
    print("This may take time for a 23-hour stream.")

    run_command([
        "yt-dlp",
        "--no-playlist",
        "--no-warnings",
        "--retries", "10",
        "--fragment-retries", "10",
        "-f", "ba",
        "-o", str(AUDIO_TEMPLATE),
        YOUTUBE_URL
    ])

    AUDIO = find_audio()

if AUDIO is None or not AUDIO.exists():
    raise RuntimeError("Audio download failed.")

print("\nAudio ready:", AUDIO)

TOTAL_DURATION = get_duration(AUDIO)

print(
    "Video duration:",
    round(TOTAL_DURATION / 3600, 2),
    "hours"
)

if TOTAL_DURATION < CLIP_DURATION:
    raise RuntimeError("Video is shorter than the requested clip duration.")


# ============================================================
# 7. LOAD WHISPER MODEL
# ============================================================

if torch.cuda.is_available():
    DEVICE = "cuda"
    COMPUTE = "float16"

    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    COMPUTE = "int8"

    print("WARNING: GPU unavailable. CPU processing will be slower.")

print("\nLoading Whisper model:", WHISPER_MODEL)

model = WhisperModel(
    WHISPER_MODEL,
    device=DEVICE,
    compute_type=COMPUTE
)


# ============================================================
# 8. RESUME OR START TRANSCRIPTION
# ============================================================

all_segments = []
completed_chunks = set()

if CHECKPOINT.exists():

    with open(CHECKPOINT, "r", encoding="utf-8") as f:
        old_checkpoint = json.load(f)

    # New checkpoints contain the source URL.
    # Legacy checkpoints are accepted for this same configured video.
    old_url = old_checkpoint.get("source_url")

    if old_url is None or old_url == YOUTUBE_URL:
        all_segments = old_checkpoint.get("segments", [])
        completed_chunks = set(
            old_checkpoint.get("completed_chunks", [])
        )

        print("Resuming saved transcription...")
    else:
        print("Checkpoint belongs to another video. Starting fresh.")

chunk_seconds = CHUNK_MINUTES * 60

total_chunks = int(
    (TOTAL_DURATION + chunk_seconds - 1) // chunk_seconds
)

print("Total chunks:", total_chunks)
print("Already completed:", len(completed_chunks))


# ============================================================
# 9. TRANSCRIBE AUDIO IN CHUNKS
# Save checkpoint after every completed chunk.
# ============================================================

for chunk_index in range(total_chunks):

    if chunk_index in completed_chunks:
        continue

    offset = chunk_index * chunk_seconds

    length = min(
        chunk_seconds,
        TOTAL_DURATION - offset
    )

    chunk_file = WORK / f"chunk_{chunk_index:04d}.wav"

    print(
        f"\nTranscribing chunk "
        f"{chunk_index + 1}/{total_chunks}"
    )

    # Extract audio section.
    run_command([
        "ffmpeg", "-y", "-v", "error",
        "-ss", str(offset),
        "-i", str(AUDIO),
        "-t", str(length),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(chunk_file)
    ])

    # Speech recognition.
    segments, info = model.transcribe(
        str(chunk_file),
        beam_size=1,
        vad_filter=True,
        condition_on_previous_text=False
    )

    for seg in segments:
        all_segments.append({
            "start": round(seg.start + offset, 3),
            "end": round(seg.end + offset, 3),
            "text": seg.text.strip()
        })

    completed_chunks.add(chunk_index)

    # Save progress to Google Drive.
    save_checkpoint({
        "source_url": YOUTUBE_URL,
        "duration": TOTAL_DURATION,
        "segments": all_segments,
        "completed_chunks": sorted(completed_chunks)
    })

    chunk_file.unlink(missing_ok=True)

    print(
        "Checkpoint saved:",
        len(completed_chunks),
        "/",
        total_chunks
    )

print("\nTranscription completed!")

if not all_segments:
    raise RuntimeError(
        "No speech was detected. Check the audio and transcription."
    )


# ============================================================
# 10. SELECT BEST SPEECH MOMENTS
# This is an automatic speech-density ranking heuristic.
# ============================================================

print("\nAnalyzing transcript...")

all_segments.sort(key=lambda x: x["start"])

starts = [s["start"] for s in all_segments]

candidates = []

STEP = 15
MIN_WORDS = 15

last_start = max(
    1,
    int(TOTAL_DURATION - CLIP_DURATION) + 1
)

for start in range(0, last_start, STEP):

    end = start + CLIP_DURATION

    index = bisect_left(starts, start)
    left = max(0, index - 5)

    words = 0
    speech_time = 0

    for seg in all_segments[left:]:

        if seg["start"] >= end:
            break

        if seg["end"] <= start:
            continue

        overlap = (
            min(seg["end"], end)
            - max(seg["start"], start)
        )

        if overlap <= 0:
            continue

        words += len(seg["text"].split())
        speech_time += overlap

    if words < MIN_WORDS:
        continue

    density = words / CLIP_DURATION
    coverage = min(1, speech_time / CLIP_DURATION)

    score = density * 0.8 + coverage * 0.2

    candidates.append({
        "start": start,
        "end": end,
        "score": score
    })

candidates.sort(
    key=lambda x: x["score"],
    reverse=True
)

top_clips = []

for candidate in candidates:

    if len(top_clips) >= min(TOTAL_CLIPS, 10):
        break

    overlaps = any(
        candidate["start"] < selected["end"]
        and candidate["end"] > selected["start"]
        for selected in top_clips
    )

    if not overlaps:
        top_clips.append(candidate)

top_clips.sort(key=lambda x: x["start"])

print("Selected clips:", len(top_clips))

if not top_clips:
    raise RuntimeError(
        "No suitable speech moments found. "
        "Try lowering MIN_WORDS in the selection section."
    )


# ============================================================
# 11. DOWNLOAD AND EDIT SELECTED VIDEO CLIPS
# ============================================================

output_files = []

for i, clip in enumerate(top_clips, 1):

    print(f"\nProcessing clip {i}/{len(top_clips)}")

    start = clip["start"]
    end = clip["end"]

    raw_template = WORK / f"raw_{i:02d}.%(ext)s"
    srt_file = WORK / f"subtitle_{i:02d}.srt"

    output = SAVE / f"Viral_Short_{i:02d}.mp4"

    if output.exists() and output.stat().st_size > 100000:
        print("Already completed:", output.name)
        output_files.append(output)
        continue

    # Remove leftovers from a previous failed attempt.
    for old_file in glob.glob(str(WORK / f"raw_{i:02d}.*")):
        Path(old_file).unlink(missing_ok=True)

    print("Downloading selected video section...")

    try:
        run_command([
            "yt-dlp",
            "--no-playlist",
            "--no-warnings",
            "--retries", "5",
            "--fragment-retries", "5",
            "--force-keyframes-at-cuts",
            "--download-sections",
            f"*{start:.3f}-{end:.3f}",
            "-f",
            f"bv*[height<={VIDEO_HEIGHT}]+ba/"
            f"b[height<={VIDEO_HEIGHT}]/best",
            "--merge-output-format", "mp4",
            "-o", str(raw_template),
            YOUTUBE_URL
        ])

    except Exception as e:
        print("Video download failed:", str(e)[:1000])
        continue

    raw_files = sorted(
        glob.glob(str(WORK / f"raw_{i:02d}.*"))
    )

    raw_files = [
        f for f in raw_files
        if not f.endswith(".part")
        and not f.endswith(".ytdl")
    ]

    if not raw_files:
        print("No video file found. Skipping.")
        continue

    raw_video = raw_files[0]


    # ========================================================
    # 12. CREATE SUBTITLES
    # ========================================================

    with open(srt_file, "w", encoding="utf-8") as f:

        subtitle_index = 1

        for seg in all_segments:

            if seg["end"] <= start or seg["start"] >= end:
                continue

            relative_start = max(0, seg["start"] - start)
            relative_end = min(
                CLIP_DURATION,
                seg["end"] - start
            )

            if relative_end <= relative_start:
                continue

            f.write(
                f"{subtitle_index}\n"
                f"{srt_time(relative_start)} --> "
                f"{srt_time(relative_end)}\n"
                f"{seg['text']}\n\n"
            )

            subtitle_index += 1


    # ========================================================
    # 13. VERTICAL FORMAT + BURNED-IN CAPTIONS
    # ========================================================

    subtitle_path = srt_file.as_posix()

    video_filter = (
        "scale=720:1280:force_original_aspect_ratio=increase,"
        "crop=720:1280,"
        f"subtitles=filename={subtitle_path}:"
        "force_style='FontName=Arial,"
        "FontSize=18,"
        "Bold=1,"
        "PrimaryColour=&H00FFFFFF,"
        "OutlineColour=&H00000000,"
        "BorderStyle=1,"
        "Outline=2,"
        "Shadow=1,"
        "Alignment=2,"
        "MarginV=55'"
    )

    print("Creating vertical video with captions...")

    try:

        run_command([
            "ffmpeg", "-y", "-v", "error",
            "-i", raw_video,
            "-t", str(CLIP_DURATION),
            "-vf", video_filter,
            "-c:v", "libx264",
            "-preset", "veryfast",
            "-crf", "23",
            "-c:a", "aac",
            "-b:a", "128k",
            "-movflags", "+faststart",
            str(output)
        ])

        if output.exists() and output.stat().st_size > 100000:

            output_files.append(output)

            print("SUCCESS:", output.name)
            print(
                "Stream time:",
                round(start, 2),
                "to",
                round(end, 2),
                "seconds"
            )

        else:
            print("Output file validation failed.")

    except Exception as e:
        print("Editing failed:", str(e)[:1000])

    # Remove temporary files.
    for f in raw_files:
        Path(f).unlink(missing_ok=True)

    srt_file.unlink(missing_ok=True)

    time.sleep(2)


# ============================================================
# 14. CREATE ZIP
# ============================================================

print("\nCreating final ZIP...")

# Include every successfully created clip.
output_files = [
    p for p in output_files
    if p.exists() and p.stat().st_size > 100000
]

if not output_files:
    raise RuntimeError(
        "No clips were created successfully. "
        "Check the video download errors."
    )

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    compression=zipfile.ZIP_STORED
) as archive:

    for video in output_files:
        archive.write(video, video.name)


# ============================================================
# 15. FINAL RESULTS
# ============================================================

print("\n======================================")
print("         PROCESS COMPLETED")
print("======================================")

print("Total clips:", len(output_files))
print("Saved in Google Drive:", SAVE)
print("ZIP file:", ZIP_FILE)

for video in output_files:
    print("READY:", video.name)

print("\nYour clips are saved in:")
print("/MyDrive/Stream_Clips/")

# Avoid sending an extremely large ZIP through the browser.
zip_size_gb = ZIP_FILE.stat().st_size / (1024 ** 3)

if zip_size_gb < 1.8:
    print("\nDownloading ZIP to your computer...")
    files.download(str(ZIP_FILE))
else:
    print("\nZIP is larger than 1.8 GB.")
    print("Open Google Drive and download it from Stream_Clips.")
